# Identifying Emerging Research Topics
Julien Garcia, 2017699

## About
This section identifies emerging research topics in the DBLP dataset by analyzing paper titles, abstracts, years, and venues. The goal is to discover major research themes, track how they change over time, and compare how quickly different venues adopt those topics.

## Tasks
- use TF-IDF features from paper titles and abstracts
- apply topic modeling to discover latent research themes
- track topic prevalence from 2000 to 2017
- identify topics that grew the most over time
- compare topic adoption across major venues
- use representative papers as case studies

## Motivations
- ...

## Challenges
- ...

In [12]:
"""Import Dependencies"""
import json
import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

In [13]:
"""Globals/Constants"""
DATA_PATH = './dblp_ref/'

START_YEAR = 2000
END_YEAR = 2017

MAX_FEATURES = 5000
NUM_TOPICS = 10
TOP_WORDS_PER_TOPIC = 12

# sample
SAMPLE_SIZE = 100000

RANDOM_STATE = 42

In [14]:
"""Helper/Utility Functions"""
def read_json_lines(file_path):
    """Reads the JSON file at the given path and yeilds each line (record)."""
    with open(file_path) as json_file:
        for line in json_file:
            yield json.loads(line)


def get_top_words(model, feature_names, n_words=10):
    #gets the main words for each topic
    rows = []
    for topic_index, topic_weights in enumerate(model.components_):
        top_indices = topic_weights.argsort()[::-1][:n_words]
        top_words = [feature_names[i] for i in top_indices]

        rows.append({
            "topic_id": topic_index,
            "top_words": ", ".join(top_words)
        })
    return pd.DataFrame(rows)

def label_topic(top_words):
    words = top_words.lower()
    if any(term in words for term in ["neural", "deep", "learning", "classification", "prediction"]):
        return "machine learning"
    elif any(term in words for term in ["image", "vision", "recognition", "object", "feature"]):
        return "computer vision"
    elif any(term in words for term in ["data", "mining", "query", "database", "web"]):
        return "data mining databases"
    elif any(term in words for term in ["wireless", "sensor", "routing", "mobile", "network"]):
        return "networks wireless systems"
    elif any(term in words for term in ["software", "program", "code", "testing"]):
        return "software engineering"
    elif any(term in words for term in ["security", "privacy", "attack", "encryption"]):
        return "security privacy"
    elif any(term in words for term in ["cloud", "distributed", "parallel", "computing"]):
        return "distributed cloud computing"
    elif any(term in words for term in ["social", "user", "recommendation", "online"]):
        return "social recommender systems"
    else:
        return "mixed topic"

In [15]:
"""Load Data"""
frames = []
for file_index in range(4):
    file_name = f'{ DATA_PATH }dblp-ref-{ file_index }.json'
    print(f"Loading file { file_name } ({ file_index + 1 }/4)...")
    frames.append(pd.DataFrame(read_json_lines(file_name)))

print("Concatenating Data Frames...")
papers = pd.concat(frames)
papers

Loading file ./dblp_ref/dblp-ref-0.json (1/4)...
Loading file ./dblp_ref/dblp-ref-1.json (2/4)...
Loading file ./dblp_ref/dblp-ref-2.json (3/4)...
Loading file ./dblp_ref/dblp-ref-3.json (4/4)...
Concatenating Data Frames...


,abstract,authors,n_citation,references,title,venue,year,id
0,The purpose of this study is to develop a lear...,"[Makoto Satoh, Ryo Muramatsu, Mizue Kayama, Ka...",0,"[51c7e02e-f5ed-431a-8cf5-f761f266d4be, 69b625b...",Preliminary Design of a Network Protocol Learn...,international conference on human-computer int...,2013,00127ee2-cb05-48ce-bc49-9de556b93346
1,This paper describes the design and implementa...,"[Gareth Beale, Graeme Earl]",50,"[10482dd3-4642-4193-842f-85f3b70fcf65, 3133714...",A methodology for the physically accurate visu...,visual analytics science and technology,2011,001c58d3-26ad-46b3-ab3a-c1e557d16821
2,This article applied GARCH model instead AR or...,"[Altaf Hossain, Faisal Zaman, Mohammed Nasser,...",50,"[2d84c0f2-e656-4ce7-b018-90eda1c132fe, a083a1b...","Comparison of GARCH, Neural Network and Suppor...",pattern recognition and machine intelligence,2009,001c8744-73c4-4b04-9364-22d31a10dbf1
3,NaN,"[Jea-Bum Park, Byungmok Kim, Jian Shen, Sun-Yo...",0,"[8c78e4b0-632b-4293-b491-85b1976675e6, 9cdc54f...",Development of Remote Monitoring and Control D...,,2011,00338203-9eb3-40c5-9f31-cbac73a519ec
4,NaN,"[Giovanna Guerrini, Isabella Merlo]",2,NaN,Reasonig about Set-Oriented Methods in Object ...,,1998,0040b022-1472-4f70-a753-74832df65266
...,...,...,...,...,...,...,...,...
79002,NaN,"[Hassan Charaf, Peter Ekler, Tamás Mészáros, I...",50,NaN,Mobile Platforms and Multi-Mobile Platform Dev...,Acta Cybernetica,2014,ff5ce050-ea8d-40e8-a25f-c629bed2ff9c
79003,NaN,"[Saul Blecker, Stuart D. Katz, Leora I. Horwit...",0,NaN,Comparison of Approaches for Heart Failure Cas...,,2016,ff5f5e4d-b650-496a-bfdd-91affb718488
79004,NaN,"[Dzmitry Bahdanau, Tom Bosc, Stanisław Jastrzę...",0,NaN,Learning to Compute Word Embeddings on the Fly,,2017,ff8fba62-4bf4-40cd-8555-46b8c64dddd7
79005,NaN,"[Kirsti Askedal, Leif Skiftenes Flak, Eirik Ab...",0,NaN,Reviewing Effects of ICT in Primary Healthcare...,,2017,ff90ffea-c94e-4ac5-a36a-05e1eccd6a76


In [16]:
"clean data for my topic analysis"
topic_papers = papers.copy()

# need title abstract and year for this part
topic_papers = topic_papers.dropna(subset=["title", "abstract", "year"])
# looking at 2000 to 2017
topic_papers = topic_papers[
    (topic_papers["year"] >= START_YEAR) &
    (topic_papers["year"] <= END_YEAR)
]
# keep venue for later
topic_papers["venue"] = topic_papers["venue"].fillna("").astype(str)
# combine title and abstract since both describe the paper
topic_papers["text"] = (
    topic_papers["title"].astype(str) + " " +
    topic_papers["abstract"].astype(str)
)
# remove short ones
topic_papers["text_length"] = topic_papers["text"].str.split().str.len()
topic_papers = topic_papers[topic_papers["text_length"] >= 20]
print("filtered papers:", topic_papers.shape)
topic_papers[["title", "venue", "year", "n_citation", "text_length"]].head()

KeyboardInterrupt: 

In [ ]:
"sample data"
if SAMPLE_SIZE is not None and len(topic_papers) > SAMPLE_SIZE:
    topic_sample = topic_papers.sample(
        n=SAMPLE_SIZE,
        random_state=RANDOM_STATE
    ).copy()
else:
    topic_sample = topic_papers.copy()
print("sample size:", topic_sample.shape)

In [ ]:
"make tfidf features"
vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    stop_words="english",
    lowercase=True,
    min_df=10,
    max_df=0.80
)
tfidf_matrix = vectorizer.fit_transform(topic_sample["text"])
print("tfidf shape:", tfidf_matrix.shape)

In [ ]:
"run topic model"
topic_model = NMF(
    n_components=NUM_TOPICS,
    random_state=RANDOM_STATE,
    init="nndsvda",
    max_iter=300
)
document_topic_matrix = topic_model.fit_transform(tfidf_matrix)
print("document topic shape:", document_topic_matrix.shape)

In [ ]:
"show topic words"
feature_names = vectorizer.get_feature_names_out()
topic_keywords = get_top_words(
    topic_model,
    feature_names,
    n_words=TOP_WORDS_PER_TOPIC
)
topic_keywords["topic_label"] = topic_keywords["top_words"].apply(label_topic)
topic_keywords

In [ ]:
"give each paper its strongest topic"
topic_sample["dominant_topic"] = document_topic_matrix.argmax(axis=1)
topic_sample["dominant_topic_weight"] = document_topic_matrix.max(axis=1)
topic_sample = topic_sample.merge(
    topic_keywords[["topic_id", "topic_label"]],
    left_on="dominant_topic",
    right_on="topic_id",
    how="left"
)
topic_sample[[
    "title",
    "venue",
    "year",
    "n_citation",
    "dominant_topic",
    "topic_label",
    "dominant_topic_weight"
]].head()

In [ ]:
"count topics"
topic_counts = (
    topic_sample["topic_label"]
    .value_counts()
    .reset_index()
)
topic_counts.columns = ["topic_label", "paper_count"]
topic_counts["percentage"] = (
    topic_counts["paper_count"] /
    topic_counts["paper_count"].sum() *
    100
)
topic_counts

In [ ]:
"plot topic counts"
plt.figure(figsize=(12, 6))
plt.bar(topic_counts["topic_label"], topic_counts["paper_count"])
plt.xticks(rotation=45, ha="right")
plt.xlabel("topic")
plt.ylabel("number of papers")
plt.title("topic counts")
plt.tight_layout()
plt.show()

In [ ]:
"topic counts by year"
year_topic_counts = (
    topic_sample
    .groupby(["year", "topic_label"])
    .size()
    .reset_index(name="paper_count")
)
year_totals = (
    topic_sample
    .groupby("year")
    .size()
    .reset_index(name="year_total")
)
year_topic_counts = year_topic_counts.merge(year_totals, on="year", how="left")
year_topic_counts["topic_percentage"] = (
    year_topic_counts["paper_count"] /
    year_topic_counts["year_total"] *
    100
)
year_topic_counts.head()

In [ ]:
"plot topics over time"
plt.figure(figsize=(14, 7))

for topic_label in sorted(year_topic_counts["topic_label"].unique()):
    topic_trend = year_topic_counts[
        year_topic_counts["topic_label"] == topic_label
    ]
    plt.plot(
        topic_trend["year"],
        topic_trend["topic_percentage"],
        marker="o",
        label=topic_label
    )
plt.xlabel("year")
plt.ylabel("percent of papers")
plt.title("topics over time")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
"find topics that grew the most"
early_period = topic_sample[
    (topic_sample["year"] >= 2000) &
    (topic_sample["year"] <= 2005)
]
late_period = topic_sample[
    (topic_sample["year"] >= 2012) &
    (topic_sample["year"] <= 2017)
]
early_topic_share = (
    early_period["topic_label"]
    .value_counts(normalize=True)
    .reset_index()
)
early_topic_share.columns = ["topic_label", "early_share"]
late_topic_share = (
    late_period["topic_label"]
    .value_counts(normalize=True)
    .reset_index()
)
late_topic_share.columns = ["topic_label", "late_share"]
topic_growth = topic_keywords[["topic_label"]].drop_duplicates()
topic_growth = topic_growth.merge(early_topic_share, on="topic_label", how="left")
topic_growth = topic_growth.merge(late_topic_share, on="topic_label", how="left")
topic_growth["early_share"] = topic_growth["early_share"].fillna(0)
topic_growth["late_share"] = topic_growth["late_share"].fillna(0)
topic_growth["growth_change"] = topic_growth["late_share"] - topic_growth["early_share"]
topic_growth["growth_percentage_points"] = topic_growth["growth_change"] * 100
topic_growth = topic_growth.sort_values(
    "growth_percentage_points",
    ascending=False
)
topic_growth

In [ ]:
"plot growing topics"
top_growth = topic_growth.head(8)
plt.figure(figsize=(12, 6))
plt.bar(top_growth["topic_label"], top_growth["growth_percentage_points"])
plt.xticks(rotation=45, ha="right")
plt.xlabel("topic")
plt.ylabel("growth percentage points")
plt.title("fastest growing topics")
plt.tight_layout()
plt.show()

In [ ]:
"topic lifecycle table"
topic_lifecycle_rows = []
for topic_label in sorted(year_topic_counts["topic_label"].unique()):
    trend = year_topic_counts[
        year_topic_counts["topic_label"] == topic_label
    ].copy()
    peak_row = trend.loc[trend["topic_percentage"].idxmax()]
    early_avg = trend[trend["year"].between(2000, 2005)]["topic_percentage"].mean()
    middle_avg = trend[trend["year"].between(2006, 2011)]["topic_percentage"].mean()
    late_avg = trend[trend["year"].between(2012, 2017)]["topic_percentage"].mean()
    topic_lifecycle_rows.append({
        "topic_label": topic_label,
        "peak_year": int(peak_row["year"]),
        "peak_percentage": peak_row["topic_percentage"],
        "early_avg_percentage": early_avg,
        "middle_avg_percentage": middle_avg,
        "late_avg_percentage": late_avg
    })
topic_lifecycle = pd.DataFrame(topic_lifecycle_rows)
topic_lifecycle = topic_lifecycle.sort_values(
    "late_avg_percentage",
    ascending=False
)
topic_lifecycle

In [ ]:
"look at topics by venue"
venue_counts = (
    topic_sample[topic_sample["venue"] != ""]
    ["venue"]
    .value_counts()
)
top_venues = venue_counts.head(10).index.tolist()
venue_topic_data = topic_sample[
    topic_sample["venue"].isin(top_venues)
].copy()
venue_topic_counts = (
    venue_topic_data
    .groupby(["venue", "topic_label"])
    .size()
    .reset_index(name="paper_count")
)
venue_totals = (
    venue_topic_data
    .groupby("venue")
    .size()
    .reset_index(name="venue_total")
)
venue_topic_counts = venue_topic_counts.merge(
    venue_totals,
    on="venue",
    how="left"
)
venue_topic_counts["topic_percentage"] = (
    venue_topic_counts["paper_count"] /
    venue_topic_counts["venue_total"] *
    100
)
venue_topic_counts.head()

In [ ]:
"venue topic table"
venue_topic_pivot = venue_topic_counts.pivot_table(
    index="venue",
    columns="topic_label",
    values="topic_percentage",
    fill_value=0
)
venue_topic_pivot

In [ ]:
"plot top growing topics by venue"
selected_topics = topic_growth.head(4)["topic_label"].tolist()
for selected_topic in selected_topics:
    plot_data = venue_topic_counts[
        venue_topic_counts["topic_label"] == selected_topic
    ].sort_values("topic_percentage", ascending=False)
    plt.figure(figsize=(12, 6))
    plt.bar(plot_data["venue"], plot_data["topic_percentage"])
    plt.xticks(rotation=45, ha="right")
    plt.xlabel("venue")
    plt.ylabel("percent of venue papers")
    plt.title(f"venue adoption for {selected_topic}")
    plt.tight_layout()
    plt.show()

In [ ]:
"case study papers"
case_study_rows = []
for topic_label in topic_growth.head(5)["topic_label"]:
    topic_examples = (
        topic_sample[topic_sample["topic_label"] == topic_label]
        .sort_values(
            ["dominant_topic_weight", "n_citation"],
            ascending=False
        )
        .head(3)
    )
    for _, row in topic_examples.iterrows():
        case_study_rows.append({
            "topic_label": topic_label,
            "title": row["title"],
            "venue": row["venue"],
            "year": row["year"],
            "n_citation": row["n_citation"],
            "topic_weight": row["dominant_topic_weight"]
        })
case_studies = pd.DataFrame(case_study_rows)
case_studies

In [ ]:
"case study papers"
case_study_rows = []
for topic_label in topic_growth.head(5)["topic_label"]:
    topic_examples = (
        topic_sample[topic_sample["topic_label"] == topic_label]
        .sort_values(
            ["dominant_topic_weight", "n_citation"],
            ascending=False
        )
        .head(3)
    )
    for _, row in topic_examples.iterrows():
        case_study_rows.append({
            "topic_label": topic_label,
            "title": row["title"],
            "venue": row["venue"],
            "year": row["year"],
            "n_citation": row["n_citation"],
            "topic_weight": row["dominant_topic_weight"]
        })
case_studies = pd.DataFrame(case_study_rows)
case_studies

In [ ]:
"save tables"
os.makedirs("outputs", exist_ok=True)
topic_keywords.to_csv("outputs/emerging_topic_keywords.csv", index=False)
topic_growth.to_csv("outputs/emerging_topic_growth.csv", index=False)
topic_lifecycle.to_csv("outputs/emerging_topic_lifecycle.csv", index=False)
venue_topic_pivot.to_csv("outputs/venue_topic_adoption.csv")
case_studies.to_csv("outputs/emerging_topic_case_studies.csv", index=False)
print("saved tables")